<!-- Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. -->

# MJX 04 — From MuJoCo to MJX

**MJX** is MuJoCo re-implemented in **JAX**. The same physics, but the data
structures are JAX arrays, so we can `jax.jit` (compile) and `jax.vmap`
(batch) the simulation and run thousands of environments in parallel on the
GPU.

| MuJoCo (C / numpy) | MJX (JAX) |
|---|---|
| `mujoco.MjModel` | `mjx.put_model(model)` |
| `mujoco.MjData` | `mjx.make_data(mx)` |
| `mujoco.mj_step(m, d)` | `mjx.step(mx, dx)` (jit/vmap-able) |

We confirm JAX sees the AMD GPU, run one MJX rollout, then batch thousands of
rollouts and measure throughput.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jp
import mujoco
from mujoco import mjx

print("JAX devices:", jax.devices())  # expect a RocmDevice on AMD GPU

In [ ]:
# A sphere that slides on a plane (a simple contact scene).
MJCF = """
<mujoco model="slider">
  <option timestep="0.004"/>
  <worldbody>
    <geom name="floor" type="plane" size="10 10 0.1"/>
    <body name="ball" pos="0 0 0.1">
      <freejoint/>
      <geom name="ball" type="sphere" size="0.1"/>
    </body>
  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(MJCF)
mx = mjx.put_model(model)
print("qpos dim:", model.nq, "| qvel dim:", model.nv)

In [ ]:
# One MJX rollout: give the ball an initial horizontal velocity and step.
# qvel layout for a freejoint: [vx, vy, vz, wx, wy, wz].
N_STEPS = 200

@jax.jit
def rollout(vx):
    dx = mjx.make_data(mx)
    dx = dx.replace(qvel=dx.qvel.at[0].set(vx))
    def body(dx, _):
        dx = mjx.step(mx, dx)
        return dx, dx.qpos[0]  # track world x of the ball
    dx, xs = jax.lax.scan(body, dx, None, length=N_STEPS)
    return xs

xs = rollout(2.0)
xs.block_until_ready()
plt.figure(figsize=(7, 3))
plt.plot(np.asarray(xs))
plt.xlabel("step"); plt.ylabel("ball x (m)"); plt.title("Single MJX rollout (vx=2.0)"); plt.grid(True)
plt.show()

In [ ]:
# Batch thousands of rollouts with jax.vmap and measure throughput.
batched = jax.jit(jax.vmap(rollout))

def time_batch(n):
    vxs = jp.linspace(0.5, 5.0, n)
    out = batched(vxs); out.block_until_ready()  # compile
    t = time.time(); out = batched(vxs); out.block_until_ready(); dt = time.time() - t
    return dt

sizes = [1, 256, 1024, 4096]
times = [time_batch(n) for n in sizes]
for n, dt in zip(sizes, times):
    print(f"{n:>5} envs x {N_STEPS} steps: {dt*1000:7.1f} ms  ->  {n*N_STEPS/dt/1e6:6.2f} M steps/s")

plt.figure(figsize=(7, 3))
plt.bar([str(n) for n in sizes], [n*N_STEPS/dt/1e6 for n, dt in zip(sizes, times)])
plt.ylabel("M physics steps / s"); plt.xlabel("parallel envs"); plt.title("MJX throughput on GPU")
plt.show()

**Takeaway:** a single rollout barely uses the GPU, but batching with `vmap`
drives millions of physics steps per second — this is what makes GPU RL
training (next notebooks) practical.